# Extract training data

This notebook will extract plate kinematic data from a plate model and other data from the `source_data` directory, writing the resulting dataset to a CSV file which can then be used to train the models in the following notebooks (`01*.ipynb`).

## Notebook setup

These cells set some of the important variables and definitions used throughout the notebook, based on the selected config file.

### Config

In [1]:
config_file = "config/.run_config.yml"

In [2]:
from lib.paths import PathConfigManager
pcm = PathConfigManager(config_file, notebook="00b")

# =====================
# Filestructure
# =====================

# Base directories
plate_model_dir = pcm.PLATE_MODEL_DIR
raster_data_dir = pcm.RASTER_DATA_DIR
mantle_data_dir = pcm.MANTLE_DATA_DIR
points_output_dir = pcm.POINTS_DATA_DIR

pcm.create_directories()

# Source data filepaths
deposits_filepath = pcm.DEPOSITS_PATH
regions_filepath = pcm.REGIONS_PATH

# Output filepaths
training_output_filepath = pcm.TRAINING_DATA_PATH

# =====================
# Notebook scope
# =====================

# Gates for determining scope of notebook run (i.e. which feature sets to extract)
use_features = pcm.use_features

# =====================
# Plate model
# =====================

# Plate model
plate_model_name = pcm.config["plate_model"]["plate_model_name"]
use_provided_plate_model = pcm.config["plate_model"]["use_provided_plate_model"]

# Timespan for analysis
min_time = pcm.config["timespan"]["min"]
max_time = pcm.config["timespan"]["max"]
times = range(min_time, max_time + 1)

# =====================
# Extraction parameters
# =====================

# Buffer distance (degrees) around reference features for sampling unlabelled points
buffer_distance = pcm.config["study_zone_buffer"]

# Number of unlabelled points to generate
num_unlabelled = pcm.config["num_unlabelled"]  # per timestep

# Random seed for reproducibility
random_seed = pcm.config["random_seed"]

# Number of processes to use
n_jobs = pcm.config["n_jobs"]

# Whether to overwrite previous rasters (default = False)
overwrite = pcm.config["overwrite_output"]

# Verbosity of logging output
verbose = pcm.config["verbose"]

### Imports

In [17]:
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    from gplately.tools import plate_isotherm_depth

from lib.assign_regions import assign_regions
from lib.calculate_convergence import run_calculate_convergence
from lib.check_files import (
    check_plate_model,
)
from lib.combine_point_data import combine_point_data
from lib.coregister_combined_point_data import run_coregister_combined_point_data
from lib.coregister_crustal_thickness import run_coregister_crustal_thickness
from lib.coregister_ocean_rasters import (
    extract_subducted_thickness,
    run_coregister_ocean_rasters,
)
from lib.create_study_area_polygons import run_create_study_area_polygons
from lib.erodep import calculate_erodep
from lib.generate_unlabelled_points import generate_unlabelled_points
from lib.misc import calculate_slab_flux, calculate_carbon
from lib.plate_models import get_plate_reconstruction
from lib.slab_dip import calculate_slab_dip
from lib.water import calculate_water_thickness
from lib.grid_features import features

# Suppress occasional joblib warnings
%env PYTHONWARNINGS=ignore::UserWarning
warnings.simplefilter("ignore", UserWarning)

env: PYTHONWARNINGS=ignore::UserWarning


### Local input and output files
If necessary, the plate model will be downloaded:

In [6]:
if use_provided_plate_model:
    check_plate_model(plate_model_dir, verbose=True)
    plate_model_name = None
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

In [7]:
# Seafloor age grid directory
# Filename format 'seafloor_age_{time}Ma.nc'
agegrid_dir = raster_data_dir / "SeafloorAge"

# Seafloor spreading rate directory
# Filename format 'spreading_rate_{time}Ma.nc'
spreadrate_dir = raster_data_dir / "SpreadingRate"

# Seafloor sediment thickness directory
# Filename format 'sediment_thickness_{time}Ma.nc'
sedthick_dir = raster_data_dir / "SedimentThickness"

# Seafloor carbonate sediment thickness directory
# Filename format 'carbonate_thickness_{time}Ma.nc'
carbonate_dir = raster_data_dir / "CarbonateThickness"

# Oceanic crustal CO2 density directory
# Filename format 'crustal_co2_{time}Ma.nc'
co2_dir = raster_data_dir / "CrustalCO2"

# Overriding plate thickness directory
# Filename format 'crustal_thickness_{time}Ma.nc'
crustal_thickness_dir = raster_data_dir / "CrustalThickness"

# Erosion/deposition rate directory
# Filename format 'erosion_deposition_{time}Ma.nc'
erodep_dir = raster_data_dir / "ErosionDeposition"

In [8]:
# Internal file/directory paths
subduction_data_filename = points_output_dir / "subducting_plate_data.csv"
study_area_dir = points_output_dir / "study_area_polygons"
combined_points_filename = points_output_dir / "combined_points.csv"

# Cumulative training data set
coregistered_data = None

## Generate study points

### Create study area polygons along subduction zones

Here we define our study area as all points on the overriding plate within a certain distance of the subduction zone (by default, $6 \degree, \approx 660\mathrm{km}$)

In [9]:
if overwrite or not study_area_dir.is_dir():
    run_create_study_area_polygons(
        nprocs=n_jobs,
        times=times,
        plate_reconstruction=plate_model,
        output_dir=str(study_area_dir),
        buffer_distance=buffer_distance,
        verbose=verbose,
        return_output=False,
    )

### Generate random unlabelled data points

The unlabelled set is created by generating uniformly-distributed random points within the polygons created in the previous cell. To change the number of points generated at each timestep, modify the `num_unlabelled` parameter defined earlier.

In [10]:
run_generate_points = overwrite or not combined_points_filename.is_file()
if run_generate_points:
    unlabelled = generate_unlabelled_points(
        times=times,
        input_dir=study_area_dir,
        num=num_unlabelled,
        threads=n_jobs,
        seed=random_seed,
        plate_reconstruction=plate_model,
        verbose=verbose,
    )

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
objc[64794]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.10.2.dylib (0x1552c25b0) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x14969e3d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[64795]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.10.2.dylib (0x15362a5b0) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x1475793d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[64790]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_

*^^Runtime: 30.9 mins for 400 Ma*

### Combine labelled deposit/non-deposit data with random unlabelled data

The function below wrangles the points generated in the previous cell into the same format as the deposit location data.

In [11]:
combined_points = None
if run_generate_points:
    combined_points = combine_point_data(
        deposit_data=deposits_filepath,
        unlabelled_data=unlabelled,
        plate_reconstruction=plate_model,
        study_area_dir=study_area_dir,
        min_time=min(times),
        max_time=max(times),
        n_jobs=n_jobs,
        verbose=verbose,
    )
    
    combined_points = combined_points.dropna(subset=["present_lon", "present_lat"])
    combined_points.to_csv(combined_points_filename, index=False)
    
    del unlabelled

Preparing labelled data...
Loading deposit data from: /Users/glados/Documents/Not Useless/Documents/University/2026/Honours/Data & Code/PUB-framework-Alfonso/data_source/deposits/deposits-Etherington.csv
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
objc[66803]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.10.2.dylib (0x154ed55b0) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x148e243d0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[66806]: Class QT_ROOT_LEVEL_POOL__THESE_OBJECTS_WILL_BE_RELEASED_WHEN_QAPP_GOES_OUT_OF_SCOPE is implemented in both /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt6Core.6.10.2.dylib (0x15128c5b0) and /Users/glados/opt/anaconda3/envs/prospectivity/lib/libQt5Core.5.15.15.dylib (0x1456683d

## Feature co-registration

### Subducting plate data

This cell will extract the subduction kinematics data from the plate model, along with datasets relating to the subducting oceanic plate: seafloor age, sediment and carbonate thickness, etc.
However, if this data has already been extracted by another notebook and `overwrite` has not been set to `True`, then the data will be read from that file instead.

In [12]:
if use_features('subduction'):
    subduction_data = None
    if overwrite or not subduction_data_filename.is_file():
        subduction_data = run_calculate_convergence(
            nprocs=n_jobs,
            min_time=min(times),
            max_time=max(times),
            plate_reconstruction=plate_model,
            verbose=verbose,
        )
        subduction_data = run_coregister_ocean_rasters(
            nprocs=n_jobs,
            times=times,
            input_data=subduction_data,
            agegrid_dir=agegrid_dir,
            spreadrate_dir=spreadrate_dir,
            plate_reconstruction=plate_model,
            sedthick_dir=sedthick_dir,
            carbonate_dir=carbonate_dir,
            co2_dir=co2_dir,
            verbose=verbose,
        )
        subduction_data["plate_thickness (m)"] = plate_isotherm_depth(
            subduction_data["seafloor_age (Ma)"],
            maxiter=100,
        )
        subduction_data = calculate_water_thickness(data=subduction_data)
        subduction_data = calculate_carbon(subduction_data)
        subduction_data = calculate_slab_flux(subduction_data)
        subduction_data = calculate_slab_dip(subduction_data)
        subduction_data = extract_subducted_thickness(
            subduction_data,
            plate_reconstruction=plate_model,
        )
        subduction_data["sediment_flux (m^2/yr)"] = (
            subduction_data["sediment_thickness (m)"]
            * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
        ).clip(0.0, np.inf)
        subduction_data["carbon_flux (t/m/yr)"] = (
            subduction_data["total_carbon_density (t/m^2)"]
            * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
        ).clip(0.0, np.inf)
        subduction_data["water_flux (m^2/yr)"] = (
            subduction_data["total_water_thickness (m)"]
            * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
        ).clip(0.0, np.inf)

        subduction_data.to_csv(subduction_data_filename, index=False)

### Assign subduction data to point deposit/non-deposit/unlabelled data

Here we assign the appropriate values for the subduction-related parameters (kinematics, seafloor age, etc.) to the deposit sites and random locations.

In [13]:
if use_features('subduction'):
    combined_points = pd.read_csv(combined_points_filename) if combined_points is None else combined_points
    subduction_data = pd.read_csv(subduction_data_filename) if subduction_data is None else subduction_data

    coregistered_data = run_coregister_combined_point_data(
        point_data=combined_points,
        subduction_data=subduction_data,
        n_jobs=n_jobs,
        verbose=verbose,
    )
    
    del combined_points, subduction_data

### Assign crustal thickness data to point data

This cell extracts the overriding plate thickness at each point.

In [14]:
if use_features('crustal'):
    coregistered_data = pd.read_csv(combined_points_filename) if coregistered_data is None else coregistered_data
    
    coregistered_data = run_coregister_crustal_thickness(
        point_data=coregistered_data,
        input_dir=crustal_thickness_dir,
        n_jobs=n_jobs,
        verbose=verbose,
    )

### Calculate cumulative erosion

Here we calculate the cumulative erosion experienced by each deposit/random point since its time of formation.

In [15]:
if use_features('erodep'):
    coregistered_data = pd.read_csv(combined_points_filename) if coregistered_data is None else coregistered_data
    
    coregistered_data = calculate_erodep(
        data = coregistered_data,
        input_dir=erodep_dir,
        n_jobs=n_jobs,
        column_name="erosion (m)",
        verbose=verbose,
    )

### Extract simple mantle features

Reconstruct labelled points to sample mantle model outputs at various depths. Linear interpolation is used to minimise distortion from temporal sparseness of the mantle grids.

In [21]:
features.available

['Temperature_Deviation', 'Plate_Velocity_Delta']

In [ ]:
if use_features('mantle'):
    coregistered_data = pd.read_csv(combined_points_filename) if coregistered_data is None else coregistered_data

    coregistered_data = features.extract(
        point_data=coregistered_data,
        mantle_dir=mantle_data_dir,
        plate_reconstruction=plate_model,
    )

ValueError: NaN values found in sampled mantle data. Check that all points are within the dataset bounds.

## Post-processing

### Assign data to regions

To divide the data into individual regions for the later analysis, we use the `regions_filename` defined earlier, if desired.

In [ ]:
if regions_filepath is not None and regions_filepath.is_file():
    points = gpd.GeoSeries.from_xy(
        coregistered_data["present_lon"],
        coregistered_data["present_lat"],
        index=coregistered_data.index,
    )
    coregistered_data["region"] = assign_regions(
        points,
        regions=regions_filepath,
    )
    del points

### Save to file

Finally, we write the dataset to a CSV file.

In [ ]:
coregistered_data.to_csv(training_output_filepath, index=False)

coregistered_data.groupby(["region", "label"]).size()

region          label     
East Asia       negative         9
                positive         5
                unlabelled    6892
North America   negative        45
                positive       257
                unlabelled    7555
Other           negative       203
                positive         1
                unlabelled    3659
South America   negative      1096
                positive       211
                unlabelled    5469
Southeast Asia  negative         4
                positive        55
                unlabelled    7461
Tethys          negative        20
                positive        68
                unlabelled    6026
dtype: int64